## 0. Configuration

In [ ]:
QUICK = False                          
SEEDS = [0] if QUICK else [0, 1, 2]
K_FOLDS = 5
CFG = dict(ct_eps=0.0, min_pos=5, n_boot=50 if QUICK else 300, shape_dup_thr=5, alpha=0.10)
PLUG = ["FT", "KD", "FD", "WiSE", "OE", "MPT", "RA-LDL", "HCP", "RCLP", "CRCL", "Joint"]   
OURS = "OE+CURE10"
import numpy as _np; GRID = _np.linspace(-8, 8, 801)

## 1. Setup

In [2]:
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config, audit, cure, cv, crc, frontier, certify, cat

OUT = config.OUTPUT_DIR
CACHE, RES = OUT / "cache", OUT / "results"
CV04 = RES / ("cv_quick" if QUICK else "cv")
RAW = RES / ("cv07_quick" if QUICK else "cv07")
RAW8 = RES / ("cv08_quick" if QUICK else "cv08")
RAW9 = RES / ("cv09_quick" if QUICK else "cv09")
RAW10 = RES / ("cv10_quick" if QUICK else "cv10")
need = [CV04 / "folds.csv"] + [RAW / f"legacy__s{s}.npz" for s in SEEDS] + [RAW10 / f"CRCL__s{s}__f0.npz" for s in SEEDS]
miss = [p for p in need if not p.exists()]
assert not miss, f"Run notebooks 04 and 07-10 first: {miss[0]} not found"

## 2. Label spaces, pooled labels and folds (labels only, no images)

In [3]:
M = config.MANIFEST_DIR
man1 = {s: pd.read_csv(M / f"rfmid1_{s}.csv", dtype={"ID": str}) for s in config.SPLITS}
space = json.load(open(M / "label_space.json"))
classes1 = space["rfmid1_classes"]
new2 = space["new"]
UNION = classes1 + new2
N_OLD = len(classes1)
shared_map = space["shared_map"]
SH = [UNION.index(c) for c in shared_map]
RT = [UNION.index(c) for c in space["retired"]]
NW = list(range(N_OLD, len(UNION)))
OLD = list(range(N_OLD))

def label_arrays(man, dataset):
    n = len(man)
    Y = np.zeros((n, len(UNION)), dtype=np.float32)
    Mk = np.zeros_like(Y)
    if dataset == "rfmid1":
        Y[:, OLD] = man[classes1].values
        Mk[:, OLD] = 1
    else:
        for c1, c2 in shared_map.items():
            j = UNION.index(c1)
            Y[:, j], Mk[:, j] = man[c2].values, 1
        for c2 in new2:
            j = UNION.index(c2)
            Y[:, j], Mk[:, j] = man[c2].values, 1
    return Y, Mk

folds_df = pd.read_csv(CV04 / "folds.csv", dtype={"ID": str})
man2_all = pd.concat([pd.read_csv(M / f"rfmid2_{s}.csv", dtype={"ID": str}) for s in config.SPLITS], ignore_index=True)
pool = folds_df.merge(man2_all, on=["split", "ID"], how="left", validate="one_to_one")
folds = pool["fold"].values
Yp, Mp = label_arrays(pool, "rfmid2")
Y1v, M1v = label_arrays(man1["val"], "rfmid1")
HR = UNION.index("HR") if "HR" in UNION else None
GROUPS = {"shared": SH, "new": NW, "all": SH + NW, "all_exHR": [c for c in SH + NW if c != HR]}

fp1 = pd.read_csv(CACHE / "fingerprints_rfmid1.csv", dtype={"ID": str})
fp2 = pd.read_csv(CACHE / "fingerprints_rfmid2.csv", dtype={"ID": str})
v1val_fp = man1["val"][["split", "ID"]].merge(fp1, on=["split", "ID"], how="left")
pool_fp = pool[["split", "ID"]].merge(fp2, on=["split", "ID"], how="left")
dmin, _ = audit.hamming_min(v1val_fp["phash"], pool_fp["phash"])
KEEP1V = dmin > CFG["shape_dup_thr"]
print(f"pooled RFMiD 2.0: {len(pool)} images, union {len(UNION)} = {len(SH)} shared + {len(RT)} discarded + {len(NW)} new")
print(f"anchoring set A: {KEEP1V.sum()} of {len(KEEP1V)} RFMiD 1.0 validation images")

pooled RFMiD 2.0: 748 images, union 57 = 38 shared + 7 discarded + 12 new
anchoring set A: 631 of 640 RFMiD 1.0 validation images


## 3. Helpers

In [4]:
def raw_file(model, seed, fold):
    return RAW / f"{model}__s{seed}__f{fold}.npz"

def load_raw(model, seed, fold):
    if model == "FD":
        return np.load(RAW8 / f"FD__s{seed}__f{fold}.npz")
    if model in ("FT", "WiSE", "OE"):
        return np.load(raw_file(model, seed, fold))
    if model in ("HCP", "RCLP", "CRCL"):
        return np.load(RAW10 / f"{model}__s{seed}__f{fold}.npz")
    return np.load(RAW9 / f"{model}__s{seed}__f{fold}.npz")

def bootstrap2(FS, curve_models, points, seeds, n_img, contrasts, n_boot, rs=0):
    est_g, est_x, est_y = certify.envelope_stats(FS, curve_models, points, seeds, GRID)
    rng = np.random.default_rng(rs)
    G, X, Yb = [], [], []
    for _ in range(n_boot):
        w = np.bincount(rng.integers(0, n_img, n_img), minlength=n_img).astype(float)
        g, x, y = certify.envelope_stats(FS, curve_models, points, seeds, GRID, w)
        G.append(g); X.append(x); Yb.append(y)
    G, X, Yb = pd.DataFrame(G), pd.DataFrame(X), pd.DataFrame(Yb)
    gap = {p: certify.ci(est_g[p], G[p]) for p in points}
    cx = {k: certify.ci(est_x[a] - est_x[b], X[a] - X[b]) for k, (a, b) in contrasts.items()}
    cy = {k: certify.ci(est_y[a] - est_y[b], Yb[a] - Yb[b]) for k, (a, b) in contrasts.items()}
    return gap, cx, cy

In [ ]:
VAR, LEG, CATLOG = {}, {}, []
CLS = SH + NW
for seed in SEEDS:
    z = np.load(RAW / f"legacy__s{seed}.npz")
    L_old, thr_old, L1v_old = z["L"], z["thr"], z["L1v"]
    LEG[seed] = (L_old, thr_old)
    fallback = np.where(np.arange(len(UNION)) < N_OLD, thr_old, 0.5)
    VAR[("Legacy", seed)] = (L_old, np.broadcast_to(thr_old, L_old.shape).copy())
    acc = {}
    for k in range(K_FOLDS):
        te = np.where(folds == k)[0]
        va = np.where(folds == (k + 1) % K_FOLDS)[0]
        tr = np.where((folds != k) & (folds != (k + 1) % K_FOLDS))[0]
        Yv, Mv, lo_va = Yp[va], Mp[va], L_old[va]
        gamma = cat.class_prevalence(Yp[tr], Mp[tr], CLS)              
        ft = load_raw("FT", seed, k)
        Ls_va, _ = cure.per_class_select(ft["Lva"], lo_va, Yv, Mv, ft["Lva"], lo_va, SH, RT)
        Ls_te, _ = cure.per_class_select(ft["Lva"], lo_va, Yv, Mv, ft["Lte"], L_old[te], SH, RT)
        acc.setdefault("Select", []).append((te, Ls_te, cv.plain_thresholds(Ls_va, Yv, Mv, fallback)))
        for m in PLUG:
            try:
                r = load_raw(m, seed, k)
            except FileNotFoundError:
                continue
            Lva, Lte, L1v = r["Lva"], r["Lte"], r["L1v"]
            plain = cv.plain_thresholds(Lva, Yv, Mv, fallback)
            acc.setdefault(m, []).append((te, Lte, plain))
            # CURE at alpha = 10% (as notebook 10)
            s = certify.fit_shape("ct", L1v[KEEP1V], Y1v[KEEP1V], M1v[KEEP1V], L1v_old[KEEP1V], thr_old, SH, fallback, None, CFG["ct_eps"])
            base = plain.copy(); base[SH] = s[SH]
            caught_v = crc.legacy_caught(lo_va, thr_old, Yv, Mv, SH)
            lam, _, _ = crc.calibrate(Lva, base, caught_v, SH, CFG["alpha"], GRID)
            acc.setdefault(f"{m}+CURE10", []).append((te, Lte, crc.shift_thresholds(base, SH, lam)))
            # CAT (inductive, on V) and CAT-T (transductive, on the test fold's scores)
            t_cat, kept = cat.cat_thresholds(Lva, gamma, plain, CLS, Mv)
            t_catt, _ = cat.cat_thresholds(Lte, gamma, plain, CLS, Mp[te])
            acc.setdefault(f"{m}+CAT", []).append((te, Lte, t_cat))
            acc.setdefault(f"{m}+CAT-T", []).append((te, Lte, t_catt))
            CATLOG.append({"method": m, "seed": seed, "fold": k, "n_fallback": len(kept),
                           "fallback_classes": ",".join(UNION[c] for c in kept)})
    for name, parts in acc.items():
        if len(parts) != K_FOLDS:
            print(f"  skipped {name} (seed {seed}): {len(parts)} of {K_FOLDS} folds")
            continue
        L = np.zeros((len(pool), len(UNION)), np.float32)
        T = np.zeros_like(L)
        for te, Lte, t in parts:
            L[te], T[te] = Lte, t[None, :]
        VAR[(name, seed)] = (L, T)
catlog = pd.DataFrame(CATLOG)
print("classes without a CAT threshold per run (no positive training label):",
      catlog.groupby("fold")["n_fallback"].mean().round(1).to_dict())
NAMES = [n for n in ["Legacy", "Select"] + [f"{m}{s}" for m in PLUG for s in ("", "+CURE10", "+CAT", "+CAT-T")]
         if all((n, s) in VAR for s in SEEDS)]
print(len(NAMES), "rows")

classes without a CAT threshold per run (no positive training label): {0: 4.0, 1: 3.0, 2: 4.0, 3: 3.0, 4: 4.0}
46 rows


## 5. Measures

In [6]:
rows = []
blocks = [np.where(folds == k)[0] for k in range(K_FOLDS)]
for (name, seed), (L, T) in VAR.items():
    if name not in NAMES:
        continue
    L_old, thr_old = LEG[seed]
    res = cv.pooled_metrics(L, T, L_old, thr_old, Yp, Mp, GROUPS, SH, CFG["min_pos"])
    res.pop("per_class_ap")
    res["NFR-_%"] = frontier.point(frontier.flip_sets(L, T, L_old, thr_old, Yp, Mp, SH))[0]
    res["img_risk_%"] = 100 * certify.pooled_image_risk(L, T, crc.legacy_caught(L_old, thr_old, Yp, Mp, SH), SH, blocks)
    res.update(cat.multilabel_f1(L, T, Yp, Mp, SH + NW))
    rows.append(dict(res, variant=name, seed=seed))
KEYS = ["mAP_shared", "mAP_all", "NFR+_%", "NFR-_%", "img_risk_%", "CF1_%", "OF1_%", "HL_%", f"regressed_classes_min{CFG['min_pos']}"]
df = pd.DataFrame(rows)
mean = df.groupby("variant")[KEYS].mean().reindex(NAMES)
sd = df.groupby("variant")[KEYS].std().reindex(NAMES)
mean.round(3).to_csv(RES / "cat13_table.csv")
display(mean.round(2))

,mAP_shared,mAP_all,NFR+_%,NFR-_%,img_risk_%,CF1_%,OF1_%,HL_%,regressed_classes_min5
variant,,,,,,,,,
Legacy,0.18,0.14,0.00,0.00,0.00,13.07,4.83,31.57,0.00
Select,0.33,0.33,10.91,4.12,34.31,25.68,23.27,7.45,1.00
FT,0.35,0.35,11.47,6.72,36.03,25.64,20.10,9.24,0.67
FT+CURE10,0.35,0.35,2.94,21.57,8.75,18.11,13.20,22.51,0.67
FT+CAT,0.35,0.35,11.30,1.69,35.73,29.92,35.53,4.00,0.67
FT+CAT-T,0.35,0.35,11.30,1.54,35.63,31.09,37.34,3.79,0.67
KD,0.36,0.36,9.85,6.17,31.43,27.15,19.64,9.82,0.33
KD+CURE10,0.36,0.36,2.94,13.92,8.77,23.44,17.67,17.32,0.33
KD+CAT,0.36,0.36,10.06,1.25,32.14,32.73,38.28,3.74,0.33


## 6. Envelope gap and paired contrasts

In [7]:
CURVES = [m for m in PLUG + ["Select"] if m in NAMES]
POINTS = [n for n in NAMES if n != "Legacy"]
FS = {(n, s): frontier.flip_sets(*VAR[(n, s)], LEG[s][0], LEG[s][1], Yp, Mp, SH) for n in POINTS for s in SEEDS}
CON = {}
for m in PLUG:
    for v in ("CAT", "CAT-T"):
        if f"{m}+{v}" in NAMES:
            CON[f"{m}+{v} vs {m}"] = (f"{m}+{v}", m)
            CON[f"{m}+CURE10 vs {m}+{v}"] = (f"{m}+CURE10", f"{m}+{v}")
t0 = time.time()
gap, cx, cy = bootstrap2(FS, CURVES, POINTS, SEEDS, len(pool), CON, CFG["n_boot"], rs=0)
print(f"bootstrap: {(time.time() - t0) / 60:.1f} min")
con = pd.DataFrame({"d_NFR+ [est, lo, hi]": cy, "d_NFR- [est, lo, hi]": cx})
display(con)
display(pd.Series({n: gap[n] for n in POINTS if "CAT" in n or n == OURS}, name="gap to envelope").to_frame())

bootstrap: 1.2 min


,"d_NFR+ [est, lo, hi]","d_NFR- [est, lo, hi]"
FT+CAT vs FT,"[-0.171, -1.007, 0.725]","[-5.035, -5.222, -4.856]"
FT+CURE10 vs FT+CAT,"[-8.355, -9.849, -7.091]","[19.882, 19.282, 20.49]"
FT+CAT-T vs FT,"[-0.171, -1.103, 0.684]","[-5.184, -5.379, -5.006]"
FT+CURE10 vs FT+CAT-T,"[-8.355, -9.847, -7.077]","[20.031, 19.424, 20.615]"
KD+CAT vs KD,"[0.213, -0.474, 0.967]","[-4.915, -5.08, -4.761]"
KD+CURE10 vs KD+CAT,"[-7.118, -8.44, -5.734]","[12.663, 12.374, 12.957]"
KD+CAT-T vs KD,"[0.554, -0.255, 1.534]","[-4.996, -5.17, -4.848]"
KD+CURE10 vs KD+CAT-T,"[-7.46, -8.78, -6.108]","[12.744, 12.45, 13.066]"
FD+CAT vs FD,"[0.895, 0.146, 1.577]","[-5.009, -5.17, -4.84]"
FD+CURE10 vs FD+CAT,"[-8.142, -9.519, -6.871]","[14.004, 13.526, 14.414]"


,gap to envelope
FT+CAT,"[-0.961, -1.694, 0.311]"
FT+CAT-T,"[-1.212, -2.022, -0.044]"
KD+CAT,"[-3.239, -4.115, -2.023]"
KD+CAT-T,"[-3.197, -4.12, -1.79]"
FD+CAT,"[-2.86, -3.703, -1.721]"
FD+CAT-T,"[-3.232, -4.257, -2.063]"
WiSE+CAT,"[-4.693, -5.529, -3.459]"
WiSE+CAT-T,"[-4.7, -5.481, -3.336]"
OE+CURE10,"[-7.73, -8.862, -6.34]"
OE+CAT,"[-4.009, -4.908, -2.713]"


In [8]:
def fmt(n, k, d=2):
    return f"{mean.loc[n, k]:.{d}f}"

print("% Table IV rows (OE)")
for n, lab, shape in [("OE+CAT", "CAT thresholds \\cite{xie2026cap}", "RFMiD 2.0 train folds"),
                      ("OE+CAT-T", "CAT thresholds, test fold", "RFMiD 2.0 train folds")]:
    if n in mean.index:
        print(f"{lab} & {shape} & none & {fmt(n, 'NFR+_%')} & {fmt(n, 'NFR-_%')} & {fmt(n, 'img_risk_%')} & & & \\\\")
print()
print("% CAT on every update: NFR+, NFR-, rho, CF1, OF1, gap to envelope")
for m in PLUG:
    n = f"{m}+CAT"
    if n in mean.index:
        print(f"{m} & {fmt(n, 'NFR+_%')} & {fmt(n, 'NFR-_%')} & {fmt(n, 'img_risk_%')} & {fmt(n, 'CF1_%', 1)} & {fmt(n, 'OF1_%', 1)} & {gap[n][0]:.2f} \\\\")

compact = lambda n: {k: round(float(mean.loc[n, k]), 3) for k in KEYS} | {"sd_NFR+": round(float(sd.loc[n, "NFR+_%"]), 3),
                                                                     "sd_NFR-": round(float(sd.loc[n, "NFR-_%"]), 3)}
send = {"notebook": "13_cat_baseline", "quick_mode": QUICK, "seeds": SEEDS, "n_boot": CFG["n_boot"],
        "means": {n: compact(n) for n in NAMES}, "gap": {n: gap[n] for n in POINTS},
        "d_NFR+": cy, "d_NFR-": cx,
        "cat_fallback_classes_per_run": float(catlog["n_fallback"].mean()),
        "cat_fallback_examples": catlog.loc[catlog["n_fallback"] > 0, "fallback_classes"].value_counts().head(5).to_dict()}
(RES / "cat13_summary.json").write_text(json.dumps(send, indent=1, default=str))
print("=" * 30, "SEND THIS BACK", "=" * 30)
print(json.dumps(send, default=str))

% Table IV rows (OE)
CAT thresholds \cite{xie2026cap} & RFMiD 2.0 train folds & none & 9.51 & 1.22 & 30.07 & & & \\
CAT thresholds, test fold & RFMiD 2.0 train folds & none & 9.42 & 1.09 & 29.72 & & & \\

% CAT on every update: NFR+, NFR-, rho, CF1, OF1, gap to envelope
FT & 11.30 & 1.69 & 35.73 & 29.9 & 35.5 & -0.96 \\
KD & 10.06 & 1.25 & 32.14 & 32.7 & 38.3 & -3.24 \\
FD & 10.49 & 1.20 & 33.35 & 31.4 & 38.3 & -2.86 \\
WiSE & 9.12 & 1.09 & 28.34 & 28.2 & 35.7 & -4.69 \\
OE & 9.51 & 1.22 & 30.07 & 30.5 & 36.9 & -4.01 \\
MPT & 10.40 & 1.27 & 32.84 & 32.6 & 38.4 & -2.64 \\
RA-LDL & 11.76 & 1.50 & 37.07 & 29.6 & 28.6 & -0.81 \\
HCP & 10.95 & 1.78 & 34.49 & 30.0 & 36.5 & -1.15 \\
RCLP & 11.38 & 1.49 & 36.12 & 30.0 & 36.3 & -1.24 \\
CRCL & 9.85 & 1.09 & 30.88 & 26.8 & 34.4 & -3.97 \\
Joint & 10.83 & 1.63 & 34.54 & 31.5 & 36.8 & -1.52 \\
============================== SEND THIS BACK ==============================
{"notebook": "13_cat_baseline", "quick_mode": false, "seeds": [0, 1, 2], "n_boo

In [ ]:

def _f1(L, T, cols):
    P = (1 / (1 + np.exp(-L[:, cols]))) >= T[:, cols]
    y, k = Yp[:, cols] == 1, Mp[:, cols] > 0
    tp, fp, fn = (P & y & k).sum(0), (P & ~y & k).sum(0), (~P & y & k).sum(0)
    has = (y & k).sum(0) > 0
    f1 = np.where(2*tp+fp+fn > 0, 2*tp / np.maximum(2*tp+fp+fn, 1), 0.0)
    return 100*f1[has].mean(), 100*2*tp.sum()/max(1, 2*tp.sum()+fp.sum()+fn.sum()), 100*((P != y) & k).sum()/max(1, k.sum())
F = []
for (name, seed), (L, T) in VAR.items():
    if name not in NAMES:
        continue
    cs, os_, hs = _f1(L, T, SH)
    ca, oa, ha = (np.nan,)*3 if name == "Legacy" else _f1(L, T, SH + NW)
    F.append(dict(variant=name, seed=seed, CF1_sh=cs, OF1_sh=os_, HL_sh=hs, CF1_all=ca, OF1_all=oa, HL_all=ha))
f1tab = pd.DataFrame(F).groupby("variant").mean(numeric_only=True).drop(columns="seed").reindex(NAMES).round(2)
print(f1tab.to_json(orient="index"))

{"Legacy":{"CF1_sh":16.51,"OF1_sh":12.57,"HL_sh":11.93,"CF1_all":null,"OF1_all":null,"HL_all":null},"Select":{"CF1_sh":27.9,"OF1_sh":28.09,"HL_sh":7.02,"CF1_all":25.68,"OF1_all":23.27,"HL_all":7.45},"FT":{"CF1_sh":27.86,"OF1_sh":23.3,"HL_sh":9.34,"CF1_all":25.64,"OF1_all":20.1,"HL_all":9.24},"FT+CURE10":{"CF1_sh":18.14,"OF1_sh":13.96,"HL_sh":26.45,"CF1_all":18.11,"OF1_all":13.2,"HL_all":22.51},"FT+CAT":{"CF1_sh":32.12,"OF1_sh":42.49,"HL_sh":3.61,"CF1_all":29.92,"OF1_all":35.53,"HL_all":4.0},"FT+CAT-T":{"CF1_sh":33.56,"OF1_sh":44.9,"HL_sh":3.37,"CF1_all":31.09,"OF1_all":37.34,"HL_all":3.79},"KD":{"CF1_sh":29.67,"OF1_sh":22.52,"HL_sh":10.05,"CF1_all":27.15,"OF1_all":19.64,"HL_all":9.82},"KD+CURE10":{"CF1_sh":24.89,"OF1_sh":19.45,"HL_sh":19.72,"CF1_all":23.44,"OF1_all":17.67,"HL_all":17.32},"KD+CAT":{"CF1_sh":35.07,"OF1_sh":45.74,"HL_sh":3.31,"CF1_all":32.73,"OF1_all":38.28,"HL_all":3.74},"KD+CAT-T":{"CF1_sh":34.53,"OF1_sh":46.34,"HL_sh":3.2,"CF1_all":32.19,"OF1_all":38.7,"HL_all":3.62},"